This notebook serves as a test for a fnl predictor on the elsner dataset. It follows the paper: "Towards detecting Primordial non-Gaussianity in the CMB using Spherical Convolutional Neural Networks" by Jorik Melsen and Thomas Floss.

# Model Tester

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
import sys
import math
import logging
import healpy as hp
import numpy as np
from matplotlib import pyplot as plt

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf

from tensorflow.keras.layers import (  # type: ignore
    Dense,
    Dropout,
    Flatten,
    LeakyReLU,
    Permute,
)
from tensorflow.keras.callbacks import (  # type: ignore
    EarlyStopping,
    TerminateOnNaN,
    TensorBoard,
    ModelCheckpoint,
)
from tensorflow.keras.optimizers import Adam, AdamW  # type: ignore
from tensorflow.keras.optimizers.schedules import ExponentialDecay  # type: ignore
from tensorflow.keras.metrics import RootMeanSquaredError  # type: ignore
from tensorflow.data.experimental import assert_cardinality
from tensorflow.keras.initializers import HeNormal

from mlpng import Core
import mlpng.utils.plots as mplt
from mlpng.utils import (
    setup_logging,
    plot_predictions,
    plot_histogram,
    print_errors,
    plot_metrics,
    try_init_wandb,
)
from mlpng.utils.dataloaders import elsnerMapDataset, elsnerDataset, MapDataset

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import (
    HealpyChebyshev,
    HealpyPool,
)

keras = tf.keras

# Setup logging for notebook
setup_logging("mlpng.notebook", level=logging.DEBUG)
logger = logging.getLogger("mlpng.notebook")

In [ ]:
print("Conda environment:", os.environ["CONDA_DEFAULT_ENV"])
print("Python executable:", sys.executable)
print(f"TensorFlow version: {tf.__version__}")
print(f"CUDA version: {tf.sysconfig.get_build_info()['cuda_version']}")
print(f"cuDNN version: {tf.sysconfig.get_build_info()['cudnn_version']}")

In [ ]:
# tf.keras.mixed_precision.set_global_policy("mixed_float16")

In [ ]:
core = Core(
    ["settings/elsner.json", "--nsims", "1000"]
)  # , "--tensorboard", "--wandb"])

In [ ]:
# train, val, test = elsnerDataset.fromCore(core, shapes=["local"]).split(
#     0.4, 0.1, 0.5, batch_size=1, to_tf=True
# )

# for alm, label in train.take(3):
#     alm = alm.numpy().astype(complex)
#     mplt.plot_cl_alm(core, alm[0], plot_camb=False, show=True, plot_func=plt.plot)

In [ ]:
# ds = elsnerMapDataset.fromCore(core, shapes=["local"])
# train, val, test = ds.split(0.4, 0.1, 0.5, to_tf=False)
# train = train.to_tf()

# for sim, label in train.take(1):
#     s = sim.numpy()[0, :, 0]
#     print(min(s), max(s))
#     hp.mollview(s, nest=True, title=f"Sim {label.numpy()[0]}")

#     # Convert the map to a power spectrum (Cl)
#     s_ring = hp.reorder(s, n2r=True)
#     cl = hp.anafast(s_ring)  # , lmax=core.lmax)
#     ell = np.arange(len(cl))
#     print("found max ell:", len(cl))

#     # Plot the power spectrum
#     plt.figure()
#     plt.plot(ell, ell * (ell + 1) / (2 * np.pi) * cl)
#     plt.xlabel(r"$\ell$")
#     plt.ylabel(r"$C_\ell$")
#     plt.title(f"Power Spectrum (Cl) for Sim {label.numpy()[0]}")
#     plt.grid(True)
#     plt.show()

In [ ]:
def get_model(input_shape, batch_size=32, n_y=1):
    nside = hp.npix2nside(input_shape[1])
    indices = np.arange(input_shape[1])
    layers = []

    n_layers = math.floor(math.log(nside, 2))
    for i in range(n_layers):
        fout = 32  # 2 ** (4 + i)
        layers.append(
            HealpyChebyshev(
                K=2,
                Fout=fout,
                # use_bias=True,
                use_bn=True,
                activation=LeakyReLU(0.3),
            )
        )
        layers.append(Dropout(0.1))
        layers.append(HealpyPool(1, "AVG"))

    layers.append(Flatten())
    layers.append(Dropout(0.3))
    layers.append(Dense(32, activation=LeakyReLU(0.3)))
    layers.append(Dense(32, activation=LeakyReLU(0.3)))
    layers.append(Dense(n_y))

    model = HealpyGCNN(
        nside,
        indices,
        layers,
        n_neighbors=8,
        max_batch_size=batch_size,
        initial_Fin=input_shape[-1],
    )

    model.build(input_shape)
    return model

In [ ]:
max_epochs = 200
batch_size = 128
shapes = ["local"]  # , "equilateral", "orthogonal"]
epoch_steps = core.total_sims * 0.4 * 25 // batch_size + 1
print("epoch_steps:", epoch_steps)
decay_steps = epoch_steps * 1

# ds = MapDataset.fromCore(core, shapes=shapes)
ds = elsnerMapDataset.fromCore(core, shapes=shapes)

# the sizes and duplicates match the paper
train, val, test = ds.split(
    0.4,
    0.1,
    0.5,
    to_tf=True,
    batch_size=batch_size,
    duplicates=[25, 10, 2],
    cache_file=f"{core.name}-nb-noltrim",
)

# the paper does not rotate the test set
test.rotate = False

# this forces generation of the test dataset to cache, and we need it later
# this way by the time the first epoch finishes the dataset has been made, allowing for manual stopping of the notebook
test_y = np.concatenate([y for _, y in test])

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    learning_rate = 5e-4
    learning_rate = ExponentialDecay(learning_rate, decay_steps, 0.95, staircase=True)
    # learning_rate = tf.keras.optimizers.schedules.CosineDecayRestarts(
    #     learning_rate, decay_steps * 3, m_mul=0.9
    # )

    model = get_model((None, core.npix, core.npols), batch_size, len(shapes))
    model.compile(
        optimizer=AdamW(learning_rate, weight_decay=0.1),  # type: ignore
        loss="mse",
        metrics=[RootMeanSquaredError()],  # type: ignore
    )

model.summary()

callbacks = [
    TerminateOnNaN(),
    EarlyStopping(
        monitor="val_loss", patience=20, restore_best_weights=True, start_from_epoch=30
    ),
]
if core.use_tb:
    callbacks.append(
        TensorBoard(
            log_dir=f"{core.dirs['tb']}/{core.name}/{core.slurm.job}",
            histogram_freq=1,
            # write_images=True,
            write_steps_per_second=True,
        )
    )
if core.use_wandb:
    try_init_wandb(
        config={
            "batch_size": batch_size,
            "max_epochs": max_epochs,
        },
        dir=core.dirs["wandb"],
        append_to=callbacks,
        patch_tb=core.use_tb,
        patch_logdir=f"{core.dirs['tb']}/{core.name}/{core.slurm.job}",
    )

history = model.fit(
    train,
    epochs=max_epochs,
    validation_data=val,
    callbacks=callbacks,
    verbose=1,
)
# model.save(f"{core.dirs['model']}/{core.name}-{core.slurm.job}-jorik.keras")

In [ ]:
logger.debug("Getting final plots")
preds = model.predict(test, verbose=1)
rmse = model.evaluate(test, verbose=1)[1]
# test_y = np.concatenate([y for _, y in test])  # type: ignore

In [ ]:
fisher = ds.get_fisher("local")
print_errors(test_y, preds, fisher)
plot_metrics(history, metrics=["loss"], show=True)
for shape in range(test_y.shape[1]):
    plot_predictions(
        test_y[:, shape],
        preds[:, shape],
        fisher=fisher,
        title=f"RMSE: {rmse:.3f}",
        show=True,
    )
    plot_histogram(test_y[:, shape], preds[:, shape], show=True)